# Phase 7 — Cross-dataset evaluation

Run on Kaggle with **T4 x2** and Internet enabled. Attach the RWF-2000, RLVS, Hockey Fights, Phase 3 paired crop-cache, Phase 5 checkpoint, and Phase 6 checkpoint/evaluation output datasets. The notebook caches RWF validation plus the two external evaluation sets, selects each model's decision threshold on RWF validation only, then evaluates RLVS and Hockey Fights without changing thresholds.

The selected Phase 6 candidate and both comparators are evaluated: X3D-S late fusion, X3D-M late fusion, and Swin3D-T paired views. Task 44 remains optional until a third dataset is selected and added to the project manifests/crop cache. Tensor caches live under `/kaggle/temp`; reports and predictions are saved under `/kaggle/working/experiments/phase7`.

## 1. Clone the current code and confirm both GPUs

In [ ]:
import os, subprocess, sys, json, zipfile
from pathlib import Path
import torch

REPO_URL = 'https://github.com/Ayaan-Ali-Khan/Violence-Detection.git'
REPO_DIR = Path('/kaggle/working/Violence-Detection')
GIT_REF = 'main'  # Pin a commit SHA for a final, reproducible report run.
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--all', '--tags'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', GIT_REF], check=True)
if GIT_REF in {'main', 'master'}:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', GIT_REF], check=True)
os.environ['PYTHONPATH'] = str(REPO_DIR / 'src')
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['MKL_NUM_THREADS'] = '2'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-kaggle.txt'], check=True)
GPU_COUNT = torch.cuda.device_count()
print('PyTorch:', torch.__version__, '| visible GPUs:', GPU_COUNT)
for gpu_id in range(GPU_COUNT):
    print(f'GPU {gpu_id}: {torch.cuda.get_device_name(gpu_id)}')
if GPU_COUNT < 2:
    raise RuntimeError('Select Kaggle T4 x2 (or another two-GPU accelerator) before running Phase 7.')


## 2. Mount data, Phase 3 crops, and checkpoints

Use the overrides below if automatic discovery finds zero or multiple matches. The crop-cache directory should end at `yolo11n` and contain `rwf2000/`, `rlvs/`, and `hockeyfights/`. The Phase 6 output input must include each `evaluation/*/test_predictions.csv` file. Checkpoints are searched in the project root `model_checkpoints/` first, then Kaggle inputs.

In [ ]:
import pandas as pd

INPUT_ROOT = Path('/kaggle/input')
# Optional path overrides. Leave None for discovery.
RWF_INPUT_DIR = None
RLVS_INPUT_DIR = None
HOCKEY_INPUT_DIR = None
CROP_CACHE_INPUT = None
X3DS_CHECKPOINT = None
X3DM_CHECKPOINT = None
SWIN_CHECKPOINT = None
PHASE6_OUTPUT_DIR = None

def unique_or_override(value, candidates, description):
    if value is not None:
        selected = Path(value)
        if not selected.exists():
            raise FileNotFoundError(f'{description} does not exist: {selected}')
        return selected
    candidates = sorted({Path(p) for p in candidates})
    print(description, 'candidates:', *candidates, sep='\n  ')
    if len(candidates) != 1:
        raise RuntimeError(f'Expected exactly one {description}; set its override above.')
    return candidates[0]

# RWF root must contain train/ and val/ class folders.
RWF_ROOT_CANDIDATES = {p.parent.parent for p in INPUT_ROOT.rglob('Fight')
    if p.is_dir() and p.parent.name == 'train'
    and (p.parent.parent / 'val' / 'Fight').is_dir()
    and (p.parent.parent / 'train' / 'NonFight').is_dir()}
RWF_ROOT = unique_or_override(RWF_INPUT_DIR, RWF_ROOT_CANDIDATES, 'RWF-2000 root')

RLVS_ROOT_CANDIDATES = {p.parent for p in INPUT_ROOT.rglob('Violence')
    if p.is_dir() and (p.parent / 'NonViolence').is_dir()
    and any(file.is_file() and file.suffix.lower() in {'.avi', '.mp4', '.mov'}
        for file in p.iterdir())}
RLVS_ROOT = unique_or_override(RLVS_INPUT_DIR, RLVS_ROOT_CANDIDATES, 'RLVS root')

HOCKEY_ROOT_CANDIDATES = {p.parent for p in INPUT_ROOT.rglob('*')
    if p.is_file() and p.suffix.lower() in {'.avi', '.mp4'}
    and p.name.lower().startswith(('fi', 'no'))}
HOCKEY_ROOT = unique_or_override(HOCKEY_INPUT_DIR, HOCKEY_ROOT_CANDIDATES, 'Hockey Fights root')

RAW_DIR = REPO_DIR / 'data' / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)
def link_dataset(name, source):
    target = RAW_DIR / name
    if target.is_symlink():
        target.unlink()
    elif target.exists():
        if any(target.iterdir()):
            raise FileExistsError(f'Inspect non-empty destination before replacing: {target}')
        target.rmdir()
    target.symlink_to(Path(source), target_is_directory=True)
    print(name, '->', target.resolve())
link_dataset('rwf2000', RWF_ROOT)
link_dataset('rlvs', RLVS_ROOT)
link_dataset('hockeyfights', HOCKEY_ROOT)

RWF_MANIFEST = REPO_DIR / 'data/splits/rwf2000.csv'
RLVS_MANIFEST = REPO_DIR / 'data/splits/rlvs_eval.csv'
HOCKEY_MANIFEST = REPO_DIR / 'data/splits/hockeyfights_eval.csv'
EVAL_MANIFESTS = {'rlvs': RLVS_MANIFEST, 'hockeyfights': HOCKEY_MANIFEST}

def crop_path_candidates(root, row):
    relative = Path(row['relative_path'])
    if relative.parts[:2] == ('data', 'raw'):
        relative = Path(*relative.parts[3:])
    name = f'{relative.name}.crop.npz'
    return [root / row['dataset'] / row['split'] / relative.parent / name,
            root / row['dataset'] / relative.parent / name]

if CROP_CACHE_INPUT is not None:
    CROP_ROOT_CANDIDATES = [Path(CROP_CACHE_INPUT)]
else:
    crop_search = INPUT_ROOT
    crop_files = list(crop_search.rglob('*.crop.npz'))
    if not crop_files:
        archives = []
        for archive_path in crop_search.rglob('*.zip'):
            try:
                with zipfile.ZipFile(archive_path) as archive:
                    if any(entry.lower().endswith('.crop.npz') for entry in archive.namelist()):
                        archives.append(archive_path)
            except zipfile.BadZipFile:
                continue
        if len(archives) == 1:
            unpacked = Path('/kaggle/working/phase3_crop_cache_unpacked')
            unpacked.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(archives[0]) as archive:
                for member in archive.infolist():
                    destination = (unpacked / member.filename).resolve()
                    if destination != unpacked.resolve() and unpacked.resolve() not in destination.parents:
                        raise ValueError(f'Unsafe crop-cache archive path: {member.filename}')
                archive.extractall(unpacked)
            crop_search = unpacked
            crop_files = list(crop_search.rglob('*.crop.npz'))
    roots = set()
    for path in crop_files:
        for dataset_name in ('rwf2000', 'rlvs', 'hockeyfights'):
            if dataset_name in path.parts:
                roots.add(Path(*path.parts[:path.parts.index(dataset_name)]))
    CROP_ROOT_CANDIDATES = sorted(roots)

MANIFEST_ROWS = {
    'rwf2000': pd.read_csv(RWF_MANIFEST).query("split == 'validation'"),
    'rlvs': pd.read_csv(RLVS_MANIFEST),
    'hockeyfights': pd.read_csv(HOCKEY_MANIFEST),
}
def crop_root_matches(root):
    return all(any(path.is_file() for path in crop_path_candidates(root, row))
        for rows in MANIFEST_ROWS.values() for row in rows.to_dict('records'))
CROP_ROOT_CANDIDATES = [root for root in CROP_ROOT_CANDIDATES if crop_root_matches(root)]
CROP_CACHE_DIR = unique_or_override(CROP_CACHE_INPUT, CROP_ROOT_CANDIDATES, 'complete Phase 3 crop-cache root')
if not crop_root_matches(CROP_CACHE_DIR):
    raise RuntimeError(f'Crop-cache override is incomplete for the required manifests: {CROP_CACHE_DIR}')
print('Crop cache root:', CROP_CACHE_DIR)

MODEL_CHECKPOINT_ROOT = REPO_DIR / 'model_checkpoints'

def find_best_checkpoint(folder_name, override):
    if override is not None:
        path = Path(override)
        if not path.is_file():
            raise FileNotFoundError(path)
        return path
    # Prefer the canonical project folder. Kaggle checkouts usually omit it
    # because model_checkpoints/ is git-ignored, so also search attached inputs.
    for search_root in (MODEL_CHECKPOINT_ROOT, INPUT_ROOT):
        matches = sorted(p for p in search_root.rglob('best.pt') if p.parent.name == folder_name)
        if len(matches) == 1:
            return matches[0]
        if len(matches) > 1:
            raise RuntimeError(f'Multiple checkpoints in {search_root}/{folder_name}: {matches}. Set its override above.')
    raise FileNotFoundError(f'No best.pt found for {folder_name} in {MODEL_CHECKPOINT_ROOT} or {INPUT_ROOT}. Set its override above.')

CHECKPOINTS = {
    'x3d_s_late_fusion': find_best_checkpoint('x3d_s_rwf2000_late_fusion', X3DS_CHECKPOINT),
    'x3d_m_late_fusion': find_best_checkpoint('x3d_m_rwf2000_late_fusion', X3DM_CHECKPOINT),
    'swin3d_t_paired_views': find_best_checkpoint('swin3d_t_rwf2000_paired_views', SWIN_CHECKPOINT),
}
print('Model checkpoints:', CHECKPOINTS)

PHASE6_REFERENCE_DIRS = {}
REFERENCE_SUFFIXES = {
    'x3d_s_late_fusion': 'x3d_s_late_fusion',
    'x3d_m_late_fusion': 'x3d_m_late_fusion',
    'swin3d_t_paired_views': 'swin3d_t_paired_views',
}
for model_name, folder_name in REFERENCE_SUFFIXES.items():
    if PHASE6_OUTPUT_DIR is not None:
        search_root = Path(PHASE6_OUTPUT_DIR)
    else:
        search_root = INPUT_ROOT
    candidates = sorted(p.parent for p in search_root.rglob('test_predictions.csv')
        if p.parent.name == folder_name and (p.parent / 'test_metrics.json').is_file())
    if len(candidates) != 1:
        raise RuntimeError(f'Expected one Phase 6 RWF test result for {model_name}; found {candidates}. Set PHASE6_OUTPUT_DIR to the Phase 6 output root.')
    PHASE6_REFERENCE_DIRS[model_name] = candidates[0]
print('RWF-2000 test references:', PHASE6_REFERENCE_DIRS)


## 3. Build and verify paired tensors with the Phase 6 preprocessing profile

All candidates consume the same 16-frame, 160-pixel full-frame plus interaction-crop input. The saved tensors are `uint8`; decoding is capped at two CPU workers per cache job. The cache includes crop-found masks so fallback rates remain auditable.

In [ ]:
INPUT_TAG = 'rwf_fullcrop_f16_s160_tau2p00_m0p10_v1'
TENSOR_ROOT = Path('/kaggle/temp/phase7_tensor_cache')
OUTPUT_ROOT = Path('/kaggle/working/experiments/phase7')
REPORT_ROOT = OUTPUT_ROOT / 'cache_verification'

def cache_and_verify(dataset_name, manifest, split):
    cache_dir = TENSOR_ROOT / dataset_name
    command = [sys.executable, 'scripts/cache_tensors.py',
        '--manifest', str(manifest), '--dataset', dataset_name,
        '--config', 'configs/preprocess_crops.yaml', '--model', 'x3d_m',
        '--cache-dir', str(cache_dir), '--crop-cache-dir', str(CROP_CACHE_DIR),
        '--mode', 'both', '--workers', '2', '--split', split]
    print('Caching:', dataset_name, split)
    subprocess.run(command, check=True)
    report_dir = REPORT_ROOT / dataset_name
    verify = [sys.executable, 'scripts/verify_tensor_cache.py',
        '--cache-dir', str(cache_dir), '--input-tag', INPUT_TAG,
        '--manifest', str(manifest), '--dataset', dataset_name,
        '--split', split, '--report-dir', str(report_dir)]
    subprocess.run(verify, check=True)
    print((report_dir / 'verification_report.md').read_text())

# Keep only the small RWF validation cache resident; each external cache is
# built, verified, evaluated, then removed to stay below Kaggle disk limits.
cache_and_verify('rwf2000', RWF_MANIFEST, 'validation')


## 4. Select fixed thresholds on RWF validation only

For each model, choose the probability threshold that maximizes violent-class F1 on the RWF train-derived validation split; ties prefer the threshold closest to 0.5. These thresholds are then reused unchanged for RLVS and Hockey Fights.

In [ ]:
MODEL_CONFIGS = {
    'x3d_s_late_fusion': 'configs/train_x3d_fusion.yaml',
    'x3d_m_late_fusion': 'configs/train_x3d_m_fusion.yaml',
    'swin3d_t_paired_views': 'configs/train_swin3d_t.yaml',
}
CALIBRATION_ROOT = OUTPUT_ROOT / 'calibration'
THRESHOLDS = {}
for model_name, config_path in MODEL_CONFIGS.items():
    prediction_dir = CALIBRATION_ROOT / model_name / 'validation'
    cache_dir = TENSOR_ROOT / 'rwf2000'
    evaluate = [sys.executable, 'src/violence_detection/evaluate.py',
        '--checkpoint', str(CHECKPOINTS[model_name]), '--config', config_path,
        '--manifest', str(RWF_MANIFEST), '--cache-dir', str(cache_dir),
        '--input-tag', INPUT_TAG, '--mode', 'both', '--include-crop-status',
        '--batch-size', '1', '--warmup-batches', '5', '--device', 'cuda:0',
        '--output-dir', str(prediction_dir), '--split', 'validation', '--threshold', '0.5']
    subprocess.run(evaluate, check=True)
    threshold_file = CALIBRATION_ROOT / model_name / 'threshold.json'
    subprocess.run([sys.executable, 'scripts/select_validation_threshold.py',
        '--predictions', str(prediction_dir / 'predictions.csv'),
        '--output', str(threshold_file)], check=True)
    THRESHOLDS[model_name] = json.loads(threshold_file.read_text())['threshold']

# Recompute the saved official RWF test decisions at these same locked
# thresholds so cross-dataset score drops use identical decision rules.
REFERENCE_OUTPUT_DIRS = {}
for model_name in MODEL_CONFIGS:
    reference_dir = OUTPUT_ROOT / 'reference_rwf_test' / model_name
    subprocess.run([sys.executable, 'scripts/apply_fixed_threshold.py',
        '--predictions', str(PHASE6_REFERENCE_DIRS[model_name] / 'test_predictions.csv'),
        '--threshold', str(THRESHOLDS[model_name]), '--output-dir', str(reference_dir)], check=True)
    REFERENCE_OUTPUT_DIRS[model_name] = reference_dir
print('Locked per-model thresholds:', THRESHOLDS)


## 5. Evaluate RLVS and Hockey Fights on both GPUs

For each dataset, evaluate model pairs concurrently on GPU 0 and GPU 1; the final model runs after the first pair. Each process uses two bounded DataLoader workers and the configured CPU thread cap. Thresholds come from RWF validation and are not changed here. Each external cache is removed after its models finish so the two large caches do not need to coexist on Kaggle disk.

In [ ]:
import shutil
import subprocess

EVAL_ROOT = OUTPUT_ROOT / 'evaluation'
EVAL_DIRS = {}

def launch_eval_group(model_names, dataset_name):
    jobs = []
    for gpu_id, model_name in enumerate(model_names):
        config_path = MODEL_CONFIGS[model_name]
        result_dir = EVAL_ROOT / model_name / dataset_name
        result_dir.mkdir(parents=True, exist_ok=True)
        EVAL_DIRS[(model_name, dataset_name)] = result_dir
        command = [sys.executable, 'src/violence_detection/evaluate.py',
            '--checkpoint', str(CHECKPOINTS[model_name]), '--config', config_path,
            '--manifest', str(EVAL_MANIFESTS[dataset_name]),
            '--cache-dir', str(TENSOR_ROOT / dataset_name), '--input-tag', INPUT_TAG,
            '--mode', 'both', '--include-crop-status', '--batch-size', '1',
            '--warmup-batches', '5', '--threshold', str(THRESHOLDS[model_name]),
            '--device', f'cuda:{gpu_id}', '--output-dir', str(result_dir),
            '--split', 'evaluation']
        log_path = OUTPUT_ROOT / 'logs' / f'{model_name}_{dataset_name}.log'
        log_path.parent.mkdir(parents=True, exist_ok=True)
        log_handle = open(log_path, 'w')
        process = subprocess.Popen(command, stdout=log_handle, stderr=subprocess.STDOUT)
        jobs.append((model_name, process, log_handle, log_path))
    failures = []
    for model_name, process, log_handle, log_path in jobs:
        return_code = process.wait()
        log_handle.close()
        print(f'{model_name}/{dataset_name}: exit={return_code}; log={log_path}')
        if return_code:
            failures.append((model_name, log_path))
        else:
            print(log_path.read_text()[-1800:])
    if failures:
        raise RuntimeError(f'Evaluation failed: {failures}')

for dataset_name, manifest in EVAL_MANIFESTS.items():
    cache_and_verify(dataset_name, manifest, 'evaluation')
    model_names = list(MODEL_CONFIGS)
    for start in range(0, len(model_names), 2):
        launch_eval_group(model_names[start:start + 2], dataset_name)
    shutil.rmtree(TENSOR_ROOT / dataset_name)
    print('Removed temporary cache after preserving reports:', dataset_name)

## 6. Compare cross-dataset metrics and inspect errors

The report computes the score drop against each matching model's RWF official test result. Hockey non-fight clips form the sports false-alarm set. The CSVs retain filenames and probabilities so high-confidence errors can be reviewed against the source videos.

In [ ]:
analysis_command = [sys.executable, 'scripts/analyze_cross_dataset_errors.py',
    '--output-dir', str(OUTPUT_ROOT / 'error_analysis')]
for model_name in MODEL_CONFIGS:
    analysis_command.extend(['--reference', f'{model_name}={REFERENCE_OUTPUT_DIRS[model_name]}'])
    for dataset_name in EVAL_MANIFESTS:
        analysis_command.extend(['--result', f'{model_name}={EVAL_DIRS[(model_name, dataset_name)]}'])
subprocess.run(analysis_command, check=True)
print((OUTPUT_ROOT / 'error_analysis' / 'cross_dataset_error_report.md').read_text())


## 7. Optional third dataset (Task 44)

No third evaluation dataset is currently selected in the project manifest, and the earlier todo item for choosing one was left open. RLVS and Hockey Fights are the two available external datasets, so this section is skipped unless a third dataset is added with a manifest and matching Phase 3 crop archives. Do not treat RWF train/validation/test clips as a third cross-dataset evaluation.

## 8. Download the Phase 7 outputs

Download `/kaggle/working/experiments/phase7/`. Keep the cache verification reports, locked thresholds, per-model RLVS/Hockey metrics and predictions, confusion matrices, and error-analysis CSVs. The multi-gigabyte temporary tensor caches are under `/kaggle/temp/phase7_tensor_cache/` and are not included in the output download.

In [ ]:
for root in [REPORT_ROOT, CALIBRATION_ROOT, OUTPUT_ROOT / 'reference_rwf_test',
             EVAL_ROOT, OUTPUT_ROOT / 'error_analysis', OUTPUT_ROOT / 'logs']:
    print(f'\n{root}')
    if root.exists():
        for path in sorted(root.rglob('*')):
            if path.is_file():
                print(path, path.stat().st_size, 'bytes')